# Per-UE Uplink DMRS Channel Estimate Extraction from Aerial Data Lake

This notebook demonstrates how to extract and visualize **pre-computed channel estimates** (H-matrices) on a per-UE basis from the Aerial Data Lake.

Unlike the `datalake_channel_estimation` notebook which re-derives H-estimates from raw IQ using pyAerial/cuPHY, this notebook reads H-estimates that cuPHY already computed during live operation and stored in ClickHouse. No GPU or pyAerial is required.

The `hest` table stores the concatenated H-estimate blob for each slot (all UE groups). The `fapi` table stores per-UE metadata including `hOffset`, `hSize`, `layerOffset`, `nSubcarriers`, `nDmrsEstimates`, and `nrOfLayers`, everything needed to extract each UE's channel matrix from the blob.

**H-estimate memory layout:** `[n_dmrs_estimates, n_subcarriers, n_bs_ants, n_layers_group]` (row-major, complex64)

**Prerequisites:** In `cuphycontroller` YAML config, enable Data Lake with at least channel estimate and pusch collections:
```yaml
data_config:
  datalake_db_write_enable: 1
  datalake_data_types: [fh, pusch, hest]
```
The `pusch` type populates the `fapi` table (per-UE metadata), and `hest` populates the `hest` table (raw H-estimate blobs). Both are required.

## Configuration, Imports, and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import clickhouse_connect

plt.rcParams['figure.figsize'] = [12, 4]

# --- Configuration ---
CLICKHOUSE_HOST = 'localhost'
NUM_BS_ANTS = 4       # Number of base station antennas

client = clickhouse_connect.get_client(host=CLICKHOUSE_HOST)

## Per-UE H-estimate Extraction

The `get_ue_hest` function extracts a single UE's channel matrix from the concatenated H-estimate blob.

For FDM (each UE in its own group), the UE's data is a contiguous block at `hOffset` with `hSize` elements. For MU-MIMO (multiple UEs sharing a group), `layerOffset` selects the UE's layers within the group matrix.

In [ ]:
def get_ue_hest(hest_blob, ue_row, n_bs_ants=NUM_BS_ANTS):
    """Extract per-UE H-estimate from the concatenated slot blob.

    Args:
        hest_blob: Flat float32 array from hest table (interleaved real/imag).
        ue_row: Row from fapi table with per-UE metadata.
        n_bs_ants: Number of base station antennas.

    Returns:
        Complex numpy array of shape (n_dmrs, n_subcarriers, n_bs_ants, n_layers_ue).
    """
    h_offset = int(ue_row['hOffset'])
    h_size = int(ue_row['hSize'])
    n_sc = int(ue_row['nSubcarriers'])
    n_dmrs = int(ue_row['nDmrsEstimates'])
    n_layers_ue = int(ue_row['nrOfLayers'])
    layer_offset = int(ue_row['layerOffset'])

    # h_size is in float2 (complex) elements; total group layers derived from dimensions
    n_layers_group = h_size // (n_dmrs * n_sc * n_bs_ants)

    # h_offset/h_size are in float2 units; multiply by 2 for float32 array indexing
    grp_ri = hest_blob[h_offset * 2 : (h_offset + h_size) * 2]
    grp = grp_ri.reshape(-1, 2).astype(np.float32).view(np.complex64).squeeze()
    grp = grp.reshape(n_dmrs, n_sc, n_bs_ants, n_layers_group)

    # layer_offset selects this UE's layers within the group (0 for FDM, >0 for MU-MIMO)
    return grp[:, :, :, layer_offset : layer_offset + n_layers_ue]

## Single-UE Channel Estimates

Query a single-UE slot and plot the channel magnitude across subcarriers for each antenna and DMRS symbol.

In [ ]:
# Pick a single-UE slot with successful CRC; join fapi and hest server-side on
# (SFN, Slot, TsTaiNs) so timestamp matching stays in ClickHouse's native types.
row_1ue = client.query_df("""
    SELECT f.SFN, f.Slot, f.rnti, f.nrOfLayers, f.rbStart, f.rbSize, f.nSubcarriers,
           f.nDmrsEstimates, f.dmrsSymbPos, f.layerOffset, f.ueGrpIdx,
           f.hOffset, f.hSize, f.sinr, f.rsrp,
           h.hestData
    FROM fapi AS f
    INNER JOIN hest AS h USING (SFN, Slot, TsTaiNs)
    WHERE f.nUEs = 1 AND f.tbCrcFail = 0 AND f.hSize > 0
    ORDER BY f.TsTaiNs
    LIMIT 1
""")

if row_1ue.empty:
    raise RuntimeError("No single-UE slot with paired hest blob found. Check that data has been collected.")

ue = row_1ue.iloc[0]
print(f"SFN/Slot: {ue['SFN']}/{ue['Slot']}  RNTI: {ue['rnti']}  "
      f"RBs: {ue['rbStart']}-{ue['rbStart']+ue['rbSize']-1}  "
      f"Layers: {ue['nrOfLayers']}  Group: {ue['ueGrpIdx']}")
print(f"hOffset: {ue['hOffset']}  hSize: {ue['hSize']}  "
      f"nSubcarriers: {ue['nSubcarriers']}  nDMRS: {ue['nDmrsEstimates']}  "
      f"SINR: {ue['sinr']:.1f} dB  RSRP: {ue['rsrp']:.1f} dB")

hest_blob = np.array(ue['hestData'], dtype=np.float32)

In [ ]:
h_ue = get_ue_hest(hest_blob, ue)
n_dmrs = h_ue.shape[0]

fig, axes = plt.subplots(1, n_dmrs, figsize=(5 * n_dmrs, 4), squeeze=False)
fig.suptitle(f"Channel Magnitude  |  RNTI {ue['rnti']}  |  SFN.Slot {ue['SFN']}.{ue['Slot']}")

for d in range(n_dmrs):
    ax = axes[0, d]
    for ant in range(NUM_BS_ANTS):
        ax.plot(np.abs(h_ue[d, :, ant, 0]), label=f'Ant {ant}')
    ax.set_title(f'DMRS {d}')
    ax.set_xlabel('Subcarrier')
    ax.set_ylabel('|H|')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Multi-UE Channel Estimates

When multiple UEs are scheduled in the same slot, cuPHY groups them by frequency allocation (FDM) or spatial multiplexing (MU-MIMO). The per-UE metadata in `fapi` provides everything needed to extract each UE's channel independently, regardless of the number of UEs.

- **FDM:** Each UE has a distinct `hOffset`/`hSize` (different frequency allocations), `layerOffset = 0`.
- **MU-MIMO:** UEs in the same group share the same `hOffset`/`hSize` but differ in `layerOffset`.

In [ ]:
# Find the first multi-UE slot and pull its fapi rows joined with the hest blob
# in a single server-side query. All returned rows share the same hestData column.
fapi_multi = client.query_df("""
    SELECT f.SFN, f.Slot, f.rnti, f.nrOfLayers, f.rbStart, f.rbSize, f.nSubcarriers,
           f.nDmrsEstimates, f.dmrsSymbPos, f.layerOffset, f.ueGrpIdx,
           f.hOffset, f.hSize, f.sinr, f.rsrp,
           h.hestData
    FROM fapi AS f
    INNER JOIN hest AS h USING (SFN, Slot, TsTaiNs)
    WHERE (f.SFN, f.Slot, f.TsTaiNs) IN (
        SELECT SFN, Slot, TsTaiNs FROM fapi
        WHERE nUEs >= 2 AND hSize > 0
        ORDER BY TsTaiNs LIMIT 1
    )
    ORDER BY f.rnti
""")

if fapi_multi.empty:
    print("No multi-UE slots with paired hest blob found.")
    hest_blob_m = None
else:
    ref_sfn = int(fapi_multi.iloc[0]['SFN'])
    ref_slot = int(fapi_multi.iloc[0]['Slot'])
    print(f"SFN/Slot: {ref_sfn}/{ref_slot}  --  {len(fapi_multi)} UEs")
    for i, ue_r in fapi_multi.iterrows():
        print(f"  UE {i}: RNTI {ue_r['rnti']}  Group {ue_r['ueGrpIdx']}  "
              f"RBs {ue_r['rbStart']}-{ue_r['rbStart']+ue_r['rbSize']-1}  "
              f"hOffset={ue_r['hOffset']}  hSize={ue_r['hSize']}  "
              f"SINR={ue_r['sinr']:.1f} dB")

    hest_blob_m = np.array(fapi_multi.iloc[0]['hestData'], dtype=np.float32)

In [ ]:
if len(fapi_multi) >= 2:
    n_ues = len(fapi_multi)
    fig, axes = plt.subplots(1, n_ues, figsize=(6 * n_ues, 4), squeeze=False)
    sfn_slot = f"{fapi_multi.iloc[0]['SFN']}.{fapi_multi.iloc[0]['Slot']}"
    fig.suptitle(f"Multi-UE Channel Magnitude  |  SFN.Slot {sfn_slot}  |  {n_ues} UEs")

    for idx in range(n_ues):
        ue_r = fapi_multi.iloc[idx]
        h = get_ue_hest(hest_blob_m, ue_r)
        ax = axes[0, idx]
        for ant in range(NUM_BS_ANTS):
            ax.plot(np.abs(h[0, :, ant, 0]), label=f'Ant {ant}')
        ax.set_title(f'RNTI {ue_r["rnti"]}  |  Group {ue_r["ueGrpIdx"]}  |  '
                     f'RBs {ue_r["rbStart"]}-{ue_r["rbStart"]+ue_r["rbSize"]-1}')
        ax.set_xlabel('Subcarrier')
        ax.set_ylabel('|H|')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## Channel Heatmap
Channel magnitude heatmap (antenna vs subcarrier) for each DMRS symbol.

In [ ]:
# Use the single-UE data from above
h_ue = get_ue_hest(hest_blob, ue)
n_dmrs = h_ue.shape[0]

fig, axes = plt.subplots(1, n_dmrs, figsize=(5 * n_dmrs, 3), squeeze=False)
fig.suptitle(f"Channel Heatmap |H|  |  RNTI {ue['rnti']}")

for d in range(n_dmrs):
    ax = axes[0, d]
    # Shape: (n_subcarriers, n_bs_ants) for layer 0
    img = ax.imshow(np.abs(h_ue[d, :, :, 0]).T, aspect='auto',
                    interpolation='nearest', cmap='viridis')
    ax.set_title(f'DMRS {d}')
    ax.set_xlabel('Subcarrier')
    ax.set_ylabel('Antenna')
    ax.set_yticks(range(NUM_BS_ANTS))
    plt.colorbar(img, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

## Channel Phase
Channel phase across subcarriers per antenna.

In [ ]:
h_ue = get_ue_hest(hest_blob, ue)

fig, ax = plt.subplots(1, 1, figsize=(12, 4))
fig.suptitle(f"Channel Phase (DMRS 0)  |  RNTI {ue['rnti']}")

for ant in range(NUM_BS_ANTS):
    ax.plot(np.angle(h_ue[0, :, ant, 0]), '.', markersize=2, label=f'Ant {ant}')

ax.set_xlabel('Subcarrier')
ax.set_ylabel('Phase (rad)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()